In [17]:

!pip -q install transformers datasets accelerate evaluate scikit-learn

In [6]:
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
import kagglehub
import pandas as pd
dataset_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Dataset loaded sucessfully")

import os
print(os.listdir(dataset_path))
csv_file = os.path.join(dataset_path, "IMDB Dataset.csv")

df = pd.read_csv(csv_file)

print(df.head())


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Dataset loaded sucessfully
['IMDB Dataset.csv']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [9]:
df['label']=df['sentiment'].map({'positive':1,'negative':0})
df=df[['review','label']]
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds,test_ds

(Dataset({
     features: ['review', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['review', 'label'],
     num_rows: 10000
 }))

In [10]:
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
  return tokenizer(batch['review'],padding=True,truncation=True,max_length=256)

train_ds=train_ds.map(tokenize,batched=True,batch_size=None)
test_ds=test_ds.map(tokenize,batched=True,batch_size=None)

print("\n Columns after tokenization")
print(train_ds.column_names)

print("first tokenized example ")
print(train_ds[0])

cols=['input_ids','attention_mask','label']
train_ds.set_format(type='torch',columns=cols)
test_ds.set_format(type='torch',columns=cols)


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]


 Columns after tokenization
['review', 'label', 'input_ids', 'attention_mask']
first tokenized example 
{'review': 'I caught this little gem totally by accident back in 1980 or \'81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn\'t stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it\'s even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! A 

In [ ]:
model=DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [52]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load('recall')
f1 = evaluate.load('f1')

In [53]:
def compute_metrics(eval_pred):
  logits,labels=eval_pred
  predictions=np.argmax(logits,axis=-1)
  return {'accuracy':accuracy.compute(predictions=predictions,references=labels)['accuracy'],
          'precision':precision.compute(predictions=predictions,references=labels)['precision'],
          'recall':recall.compute(predictions=predictions,references=labels)['recall'],
          'f1':f1.compute(predictions=predictions,references=labels)['f1'] }

In [4]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=2,
    logging_steps=100
  
)




In [5]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class = tokenizer,
    compute_metrics=compute_metrics
)

NameError: name 'model' is not defined

In [46]:
!pip install -U transformers datasets accelerate evaluate torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 40.0 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


In [2]:
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'sentiment-analysis',
    model = 'distilbert_imdb',
    tokenizer = 'distilbert_imdb'
)

print(classifier('this movie was absolutely fantanstic'))
print(classifier("Worst movie ever made"))

In [1]:
!pip uninstall -y datasets
!pip install datasets==3.1.0

Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 12.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
